# 25. एकीकृत सत्यापन: अंतिम-टोकन राउटर और विषम सिनैप्स पर सिमेंटिक फ़ॉलबैक

इस नोटबुक में, हम एसआरए के निकट-अंतिम रूप का एक एकीकृत सत्यापन करते हैं: "मदरबोर्ड आर्किटेक्चर"।
- **लास्ट-टोकन राउटर**: हल्का, उच्च-विश्वास वाला राउटर (नोटबुक 24 में इष्टतम साबित हुआ) जो सिग्नल कमजोर पड़ने से बचाता है
- **विषम सिनैप्स**: तंत्रिका (सामान्य प्रयोजन एलएलएम), नियम-आधारित (कैलकुलेटर), पुनर्प्राप्ति (वेक्टरडीबी)
- **सिमेंटिक फ़ॉलबैक**: विषम मॉड्यूल के बीच सुरक्षित री-रूटिंग का सत्यापन

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. विषम सिनैप्स को परिभाषित करना
हम मौलिक रूप से भिन्न कार्य सिद्धांतों के साथ तीन मॉड्यूल को परिभाषित करते हैं - एक तंत्रिका नेटवर्क (एलएलएम), एक नियतात्मक नियम-आधारित मॉड्यूल (कैलकुलेटर), और एक बाहरी ज्ञान लुकअप (वेक्टरडीबी) - एक ही इंटरफ़ेस के पीछे।

In [ ]:
class LLMSynapse(nn.Module):
    """General-purpose LLM synapse (Neural)"""
    def __init__(self, d_model):
        super().__init__()
        # In a real system this would contain a Transformer Block; we use a linear layer as a dummy
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model)
        )
    def forward(self, x, text_input=None):
        return self.net(x)

class CalculatorSynapse(nn.Module):
    """Rule-based calculator synapse"""
    def __init__(self):
        super().__init__()
        self.is_neural = False # non-neural flag
    def forward(self, x, text_input=None):
        # At real inference time, evaluate the arithmetic expression string
        if text_input is not None:
            try:
                # Simple expression evaluation (note the security implications)
                ans = eval(text_input)
                return f"CALC_RESULT: {ans}"
            except:
                return "CALC_ERROR"
        return x # dummy for gradient computation

class VectorDBSynapse(nn.Module):
    """Fact-retrieval synapse (Retrieval)"""
    def __init__(self, d_model):
        super().__init__()
        self.is_neural = False
        # Simple key-value store
        self.db = {}
        self.d_model = d_model
        
    def add_fact(self, key, value):
        self.db[key] = value
        
    def forward(self, x, text_input=None):
        if text_input is not None:
            # Simple exact-match lookup
            for k, v in self.db.items():
                if k in text_input:
                    return f"DB_RESULT: {v}"
            return "DB_MISS"
        return x

## 2. लास्ट-टोकन राउटर और मदरबोर्ड को लागू करना
हम नोटबुक 24 में चुने गए `Last-Token Router` के साथ SRA मदरबोर्ड को लागू करते हैं। इसमें तर्क शामिल है कि ** सुरक्षित रूप से राउटर की दूसरी पसंद पर वापस आ जाता है** जब कोई विशेष सिनैप्स अपनी क्षमता सीमा तक पहुंचता है।

In [ ]:
class LastTokenRouter(nn.Module):
    """
    The router judged optimal in Notebook 24.
    Computes routing probabilities using only the last token of the sequence.
    """
    def __init__(self, d_model, num_synapses):
        super().__init__()
        self.classifier = nn.Linear(d_model, num_synapses, bias=False)
        
    def forward(self, x):
        # x shape: (batch, seq_len, d_model)
        # Extract only the last token
        last_token_embeds = x[:, -1, :] # (batch, d_model)
        logits = self.classifier(last_token_embeds)
        probs = F.softmax(logits, dim=-1)
        return probs, logits

class SRAMotherboard(nn.Module):
    """Motherboard model that integrates the router and heterogeneous synapses"""
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        
        # Dictionary for registering synapses
        self.synapses = nn.ModuleDict()
        self.synapse_names = []
        
        # Router (None at first, since there are no synapses yet)
        self.router = None 
        
    def add_synapse(self, name, synapse_module):
        """Hot-Swap: dynamically add a new synapse"""
        self.synapses[name] = synapse_module
        self.synapse_names.append(name)
        # Rebuild the router (preserving existing weights)
        old_router = self.router
        self.router = LastTokenRouter(self.d_model, len(self.synapse_names)).to(device)
        
        if old_router is not None:
            # Copy over the old weights
            with torch.no_grad():
                self.router.classifier.weight[:len(self.synapse_names)-1, :] = old_router.classifier.weight.data
                
    def route_and_forward(self, input_ids, text_inputs=None, capacity_limits=None):
        """
        Inference with Semantic Fallback included.
        capacity_limits: per-synapse upper bounds, e.g. {'Calculator': 1}
        """
        x = self.embedding(input_ids) # (batch, seq, d_model)
        
        # 1. Compute probabilities with the router
        probs, logits = self.router(x)
        
        batch_size = x.size(0)
        final_outputs = []
        routing_history = []
        
        # Track the current usage of each synapse
        current_usage = {name: 0 for name in self.synapse_names}
        if capacity_limits is None:
            capacity_limits = {}
            
        for i in range(batch_size):
            # Sort the probability distribution of the i-th sample
            sample_probs = probs[i]
            sorted_indices = torch.argsort(sample_probs, descending=True)
            
            selected_synapse_name = None
            text_input = text_inputs[i] if text_inputs else None
            
            # Semantic Fallback: assign in descending order of probability, respecting capacity
            for rank, syn_idx in enumerate(sorted_indices):
                syn_name = self.synapse_names[syn_idx.item()]
                limit = capacity_limits.get(syn_name, float('inf'))
                
                if current_usage[syn_name] < limit:
                    # If there is remaining capacity, finalize the assignment
                    selected_synapse_name = syn_name
                    current_usage[syn_name] += 1
                    
                    if rank > 0:
                        print(f"[Fallback Triggered] Token '{text_input}': {self.synapse_names[sorted_indices[0].item()]} is full -> Re-routed to {syn_name}")
                    break
            
            # Process with the chosen synapse
            if selected_synapse_name:
                synapse = self.synapses[selected_synapse_name]
                if getattr(synapse, "is_neural", True):
                    # Neural processing
                    out = synapse(x[i:i+1], text_input=text_input)
                    final_outputs.append(f"[{selected_synapse_name}] Neural Output Generated (LLM handled it)")
                else:
                    # Rule-based / DB processing
                    out = synapse(x[i:i+1], text_input=text_input)
                    final_outputs.append(f"[{selected_synapse_name}] {out}")
            else:
                final_outputs.append("[Error] No Synapse Available (All Full)")
                
            routing_history.append((probs[i].detach().cpu().numpy(), selected_synapse_name))
            
        return final_outputs, routing_history

## 3. हॉट-स्वैपिंग सिनैप्स और डेटासेट तैयार करना

In [ ]:
# Dummy tokenizer and dimensionality
vocab_size = 1000
d_model = 128

# Initialize the model
model = SRAMotherboard(d_model=d_model, vocab_size=vocab_size).to(device)

# 1. Add the general-purpose LLM
model.add_synapse("GeneralLLM", LLMSynapse(d_model).to(device))

# 2. Hot-Swap the Calculator plugin
calc = CalculatorSynapse()
model.add_synapse("Calculator", calc)

# 3. Hot-Swap the VectorDB plugin
vdb = VectorDBSynapse(d_model)
vdb.add_fact("Japan", "Tokyo")
vdb.add_fact("CEO", "John Doe")
model.add_synapse("VectorDB", vdb)

print(f"Active Synapses: {model.synapse_names}")

# Prepare the dataset (3 domains: arithmetic, fact retrieval, general conversation)
data = [
    {"domain": "math", "text": "15 * 4", "target_idx": 1},
    {"domain": "math", "text": "100 / 2", "target_idx": 1},
    {"domain": "fact", "text": "Capital of Japan?", "target_idx": 2},
    {"domain": "fact", "text": "Who is the CEO?", "target_idx": 2},
    {"domain": "chat", "text": "Hello, how are you?", "target_idx": 0},
    {"domain": "chat", "text": "Tell me a joke.", "target_idx": 0},
]

# Generate dummy input IDs
input_ids = torch.randint(0, vocab_size, (len(data), 5)).to(device)
texts = [d["text"] for d in data]
target_labels = torch.tensor([d["target_idx"] for d in data]).to(device)

## 4. राउटर ट्रेनिंग (रूटिंग फ़ाइन-ट्यूनिंग)
हम लास्ट-टोकन राउटर को प्रशिक्षित करते हैं ताकि यह प्रत्येक इनपुट को सही विशेषज्ञ मॉड्यूल तक रूट कर सके।

In [ ]:
# Router training (Fine-Tuning)
optimizer = torch.optim.Adam(model.router.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

epochs = 100
model.train()
for epoch in range(epochs):
    optimizer.zero_grad()
    x = model.embedding(input_ids)
    
    # Get the router output
    probs, logits = model.router(x)
    
    # Cross-entropy against the target synapses
    loss = criterion(logits, target_labels)
    
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Loss: {loss.item():.4f}")

print("\n--- Training Complete ---")
model.eval()
with torch.no_grad():
    final_probs, _ = model.router(model.embedding(input_ids))
    print(f"Final Routing Probabilities for '{texts[0]}':")
    print({name: f"{p:.4f}" for name, p in zip(model.synapse_names, final_probs[0].cpu().tolist())})

## 5. सिमेंटिक फॉलबैक को मान्य करना
यहां, हमने जानबूझकर कैलकुलेटर और वेक्टरडीबी मॉड्यूल दोनों पर **क्षमता सीमा = 1** निर्धारित की है।
चूंकि बैच में प्रत्येक के लिए दो अनुरोध हैं, उनमें से एक फिट नहीं होगा।

हम जांचते हैं कि ओवरफ्लो हो रहा अनुरोध, कुछ यादृच्छिक मॉड्यूल में क्रैश होने के बजाय, **शब्दार्थ रूप से राउटर की दूसरी पसंद, सामान्य प्रयोजन एलएलएम पर वापस आ जाता है।

In [ ]:
print("=== 1. Normal inference (no Capacity Limit) ===")
outputs, history = model.route_and_forward(input_ids, text_inputs=texts)
for t, out in zip(texts, outputs):
    print(f"Input: {t:<20} | Output: {out}")

print("\n=== 2. Semantic Fallback validation (Calculator and VectorDB capacities limited to 1) ===")
# There are two requests for each, so one is guaranteed to overflow.
# If an overflowing VectorDB request were routed to the Calculator it would error out;
# with the general-purpose LLM as the second choice, the system can safely re-route.
limits = {"Calculator": 1, "VectorDB": 1}
outputs_fb, history_fb = model.route_and_forward(input_ids, text_inputs=texts, capacity_limits=limits)
print("-" * 40)
for t, out in zip(texts, outputs_fb):
    print(f"Input: {t:<20} | Output: {out}")

## 6. रूटिंग की कल्पना करना
हम हीटमैप के रूप में राउटर की संभाव्यता वितरण का निरीक्षण करते हैं।
समर्पित मॉड्यूल को उच्च विश्वास के साथ चुना जाता है, जबकि जनरलएलएलएम में अभी भी दूसरी पसंद के रूप में एक छोटा संभाव्यता द्रव्यमान है - यह दर्शाता है कि सुरक्षित सिमेंटिक फ़ॉलबैक संभव है।

In [ ]:
prob_matrix = torch.stack([torch.tensor(h[0]) for h in history_fb]).numpy()

plt.figure(figsize=(10, 6))
sns.heatmap(prob_matrix, annot=True, cmap="YlGnBu", fmt=".4f",
            xticklabels=model.synapse_names, 
            yticklabels=[f"{d['domain']}: {d['text']}" for d in data])
plt.title("Routing Probabilities (Semantic Fallback Enabled)")
plt.xlabel("Synapses")
plt.ylabel("Inputs")
plt.tight_layout()
plt.show()